In [2]:
from dotenv import load_dotenv
import os
import requests
from typing import *
import sys
import subprocess
import shlex
from datetime import datetime
import json
import uuid
import pandas as pd
from google.cloud import storage

# Add path to import custom modules
# sys.path.append(os.path.abspath("../src"))

load_dotenv()

True

In [3]:
# TEMP_DIR = "./temp/"
# RAW_DIR = os.path.join(TEMP_DIR,"raw")
# PQ_DIR = os.path.join(TEMP_DIR,"pq")

# temp_dirs = [RAW_DIR,PQ_DIR]

# for dir in temp_dirs:
#     os.makedirs(dir,exist_ok=True)

In [4]:
# Delete temp directories
# result = subprocess.Popen(f"rm -rvf {TEMP_DIR}",shell=True,text=True)
# result

In [5]:
def partition_id_by_year_quarter(p):
    return "".join(p.get('display_name').split(" ")[:2])

def partition_id_by_year(p):
    return p.get('display_name').split(" ")[0]

def no_of_parts_in_partition(p):
    return int(p.get('display_name').replace("(", "").replace(")","").split(" ")[-1])

def part_size_mb(p):
    return float(p.get('size_mb'))

def get_total_size(json):
    return round(sum([p.get('size_mb') for p in json.get('partitions')]),2)

def read_json_file(json_path):
    with open(json_path, "r") as f:
        d = json.load(f)
        return d
    
def filter_partition(years='', partitions=[]):
    years = [y.strip() for y in years.split(",")]
    if not years:
        print("No args provided")
        return
    return [p for p in partitions if p.get('partition_id') in years]

"""
Function to restructure JSON object to handle batch processing better
"""
def extract_drug_events(data):
    events = data.get('results').get('drug').get('event')
    total_records = events.get('total_records')
    partitions = events.get('partitions')

    # Generate unique partition_id and count set
    partition_ids = {}
    for p in partitions:
        id = partition_id_by_year(p)
        partition_ids[id] = partition_ids.get(id,0) + 1
    
    # Groups partition by partitionid
    results = []
    for item in partition_ids.items():
        id, count = item
        file_list = []
        counter = 0
        tot_size = 0

        for p in partitions:
            if counter == count:
                break
            if partition_id_by_year(p) == id:
                counter+=1
                file_list.append(p.get('file'))
                tot_size+=part_size_mb(p)
                
        results.append(
            {
                "partition_id": id,
                "count": count,
                "size_mb" : round(tot_size,2),
                "files" : file_list
            }
        )
    
    return {
        "total_records" : total_records,
        "partitions" : results
    }

"""
Function to seggregate partitions as batches based on disksize threshold
"""
def create_batch(partitions, max_batch_size_mb=10000):
    batch = []                  # partitions per batch
    batch_partitions = []       # Partitions under the threshold
    big_batch_partitions = []   # Different approach to process bigger partitions
    sum_size = 0                # Size counter

    for p in partitions:
        size = p.get('size_mb', 0)

        if size > max_batch_size_mb:
            # TODO:
            # Handle oversized partititions
            big_batch_partitions.append(p)
            continue
        
        if sum_size + size > max_batch_size_mb:
            # TODO:
            # - Declare batch_partitions as batch #
            # - Reset sum_size
            # - Reset batch_partitions
            batch.append(batch_partitions.copy())
            batch_partitions.clear()
            sum_size = 0
            continue
        
        batch_partitions.append(p)
        sum_size += size

    # Flush batch_partitions to schedule as last batch
    if len(batch_partitions) != 0:
        batch.append(batch_partitions.copy())
        batch_partitions.clear()
    
    return batch, big_batch_partitions

In [6]:
# Link to downloads of Drug Event data
res = requests.get("https://api.fda.gov/download.json")
data = res.json()
downloads_json = extract_drug_events(data)

partition = filter_partition(years='2004', partitions=downloads_json.get('partitions'))
partition

[{'partition_id': '2004',
  'count': 20,
  'size_mb': 1040.06,
  'files': ['https://download.open.fda.gov/drug/event/2004q3/drug-event-0001-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0002-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0003-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0004-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0005-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0001-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0002-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0003-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0004-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0005-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q1/drug-e

In [7]:
def download_file(url, download_path="tmp", filename="temp.json"):
    os.makedirs(download_path,exist_ok=True)
    subprocess.run(
        f'wget -q -O - {url} | gunzip > {os.path.join(download_path,filename)}',
        shell=True,
    )

In [ ]:
# Download partition files temporarily
for files in partition[0].get('files'):
    filename = ".".join("_".join(files.split('/')[-2:]).split('.')[:-1])
    download_file(files,filename=filename)

In [9]:
# Iterate through tmp           - DONE
# Load each json                - DONE
# Navigate to 'results' key     - DONE
# Keep key count                - DONE
# Detect Null                   - DONE
from operator import itemgetter

dict_count = {}
for f in os.listdir('tmp'):
    results = read_json_file(f"./tmp/{f}").get('results')

    for r in results:
        for k,v in r.items():
            # Detect null
            if not v:
                continue

            # Register non-null key count 
            dict_count[k] = dict_count.get(k,0) + 1

sorted(dict_count.items(), key=itemgetter(1), reverse=True)

[('safetyreportid', 214815),
 ('transmissiondateformat', 214815),
 ('transmissiondate', 214815),
 ('serious', 214815),
 ('receivedateformat', 214815),
 ('receivedate', 214815),
 ('receiptdateformat', 214815),
 ('receiptdate', 214815),
 ('fulfillexpeditecriteria', 214815),
 ('sender', 214815),
 ('patient', 214815),
 ('companynumb', 193324),
 ('primarysource', 161099),
 ('seriousnesshospitalization', 64499),
 ('seriousnessother', 64418),
 ('seriousnessdeath', 25447),
 ('seriousnesslifethreatening', 12338),
 ('seriousnessdisabling', 8178),
 ('seriousnesscongenitalanomali', 880),
 ('safetyreportversion', 50),
 ('primarysourcecountry', 50),
 ('reporttype', 50),
 ('receiver', 50),
 ('duplicate', 49),
 ('reportduplicate', 49),
 ('occurcountry', 45)]

In [ ]:
mx = max(dict_count.items(), key=lambda items: items[1])[1]

for k,v in sorted(dict_count.items(), key=lambda item: item[1], reverse=True):
    print(f"{k} : {round(v*100/mx):.2f}%")


In [ ]:
dict_unique = set()
for f in os.listdir('tmp'):
    results = read_json_file(f"./tmp/{f}").get('results')

    for r in results:
        for k,v in r.items():
            if isinstance(v,dict):
                v = "Object"
            dict_unique.add((k,v))

dict_unique

In [ ]:
from itertools import groupby
from operator import itemgetter
from pprint import pprint

sorted_set =  sorted(dict_unique, key=itemgetter(0))
grouped = {
    key: [val for _, val in group]
    for key, group in groupby(sorted_set, key=itemgetter(0))
}

pprint(grouped)

In [ ]:
test = [
    {
        "A" : 1,
        "B" : {
            "sub1" : 10,
            "sub2" : 20
        },
        "C" : [
            100,
            200
        ],
    },
    {
        "A" : 2,
        "B" : {
            "sub1" : 30,
            "sub2" : 40
        },
        "C" : [
            200,
            400
        ],
    },
]
test